# Semi-Automatic Entity Pre-Labelling — Sirah Nabawiyah

Deteksi entitas PERSON, EVENT, LOCATION, TIME menggunakan regex & keyword matching.\
**Output:** CSV pre-annotated yang tinggal direview/koreksi manual.

---
## 1.1 Import & Konfigurasi

In [1]:
import re
import pandas as pd
from pathlib import Path

# ── Konfigurasi ──────────────────────────────────────────────────────────────
IN_SEED = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\manual_labelling\sirah_manual_seed.csv")
OUT_PRE = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\manual_labelling\sirah_prelabelled.csv")

## 1.2 PERSON Patterns

Daftar nama tokoh utama + regex untuk nama dengan nasab (bin/binti).

In [2]:
PERSON_EXACT = [
    # Nabi & julukan
    "Rasulullah", "Nabi Muhammad", "Muhammad",
    # Keluarga Nabi
    "Khadijah", "Khadijah binti Khuwailid", "Aisyah",
    "Fatimah", "Ali bin Abu Thalib", "Ali bin Abi Thalib", "Ali",
    "Hamzah", "Hamzah bin Abdul Muththalib",
    "Abu Thalib", "Abbas bin Abdul Muththalib", "Al-Abbas",
    "Abdullah bin Abdul Muththalib", "Abdul Muththalib",
    "Aminah binti Wahb", "Halimah",
    "Hasan", "Husain", "Ja'far bin Abu Thalib",
    "Abu Sufyan bin Al-Harits bin Abdul Muththalib",
    # Khulafaur Rasyidin & sahabat utama
    "Abu Bakar", "Abu Bakar Ash-Shiddiq",
    "Umar bin Al-Khaththab", "Umar",
    "Utsman bin Affan", "Utsman",
    "Zaid bin Haritsah", "Zaid",
    "Bilal", "Bilal bin Rabah",
    "Salman Al-Farisi", "Salman",
    # Sahabat lain
    "Sa'd bin Mu'adz", "Sa'd bin Ubadah", "Sa'd bin Abu Waqqash",
    "Abdurrahman bin Auf", "Thalhah bin Ubaidillah",
    "Zubair bin Al-Awwam", "Abu Ubaidah bin Al-Jarrah",
    "Mush'ab bin Umair", "Mush'ab",
    "Khalid bin Al-Walid", "Khalid",
    "Amr bin Al-Ash", "Abu Hurairah",
    "Ubay bin Ka'b", "Mu'adz bin Jabal", "Mu'adz",
    "Abu Dzar", "Abu Dzar Al-Ghifari",
    "Anas bin Malik", "Jabir bin Abdullah", "Jabir",
    "Khabbab bin Al-Aratt", "Khabbab",
    "Abdullah bin Mas'ud", "Ibnu Mas'ud",
    "Abdullah bin Abbas", "Ibnu Abbas",
    "Abdullah bin Umar", "Ibnu Umar",
    "Abdullah bin Ubay", "Abdullah bin Ubay bin Salul",
    "Abdullah bin Rawahah",
    "Ubadah bin Ash-Shamit",
    "Usamah bin Zaid", "Usamah",
    "As'ad bin Zurarah", "As'ad",
    "Al-Barra' bin Ma'rur", "Al-Barra'",
    "Usaid bin Hudhair", "Usaid",
    "Ka'b bin Malik", "Ka'b",
    "Sa'd bin Ar-Rabi'",
    "Rifa'ah bin Abdul Mundzir",
    "Al-Mundzir bin Amr",
    "Sa'd bin Khaitsamah",
    "Abul Haitsam bin At-Taihan",
    "Al-Abbas bin Ubadah",
    "Abdullah bin Amr bin Haram",
    "Al-Muth'im bin Adi",
    "Al-Harits bin Harb",
    "Siba bin Arfazhah",
    "Zaid bin Tsabit",
    "Abdullah bin Abu Rabi'ah",
    # Istri Nabi lain
    "Hafshah", "Ummu Salamah", "Zainab",
    "Shafiyyah", "Juwairiyah",
    "Maimunah", "Maimunah binti Al-Harits Al-Amiriyah", "Maimunah binti Al-Harits Al- Amiriyah",
    # Musuh & tokoh Quraisy
    "Abu Jahal", "Abu Lahab", "Abu Sufyan bin Harb",
    "Utbah bin Rabi'ah", "Utbah",
    "Syaibah bin Rabi'ah",
    "Walid bin Al-Mughirah",
    "Umayyah bin Khalaf", "Umayyah",
    "Ubay bin Khalaf",
    "Hind binti Utbah",
    "Ikrimah bin Abu Jahal",
    "Suhail bin Amr",
    "Al-Akhnas bin Syariq",
    "Al-Aswad bin Al-Muththalib",
    "Amr bin Luhay",
    # Tokoh lain
    "Waraqah bin Naufal", "Waraqah",
    "Jibril", "Musa", "Ibrahim", "Isa", "Isma'il",
    "Adam", "Nuh", "Yusuf", "Harun", "Idris",
    "Yahya bin Zakaria", "Isa bin Maryam",
    "Harun bin Imran", "Musa bin Imran",
    "Dzu Nuwas", "Abrahah",
    "As'ad Abu Karib",
    "Bukhtanashar",
    "Najasyi", "An-Najasyi",
    "Heraklius", "Kisra",
    "Farwah bin Amr Al-Judzami",
    "Rabi'ah bin Umayyah",
    "Qais bin Al-Aslat",
    "Nasibah binti Ka'b",
    # Perawi & ulama yang disebut
    "Ibnu Hisyam", "Ibnu Ishaq", "Ibnu Sa'd",
    "Ibnu Hajar", "Ibnul Qayyim",
    "An-Nawawi", "Al-Qurthubi",
    "Ath-Thabari", "Al-Baihaqi",
    "Abu Dawud", "Abu Qatadah",
    "Sa'id bin Al-Musayyab",
    "As-Samhudi",
    "Al-Khadhri",
    "Ummul Khair", "Ummu Jamil",
]

# ── Improved regex untuk nama Arab dengan nasab ──────────────────────────────
# Menangani pola: Name bin Al-Something, Name bin Abdul Something, dst.
#
# Atom nama: kata kapital, opsional diawali artikel Arab (Al-, Ash-, An-, dst.)
_ART = r"(?:(?:Al|An|Ash|As|Ats|Ad|Ar|Az|At|Adz)-)"
_ATOM = r"(?:" + _ART + r")?[A-Z][a-z']+(?:'[a-z]*)?"
#
# Setelah bin/binti, nama bisa compound: "Abdul Muththalib", "Abu Thalib"
_COMPOUND = r"(?:Abdul|Abu|Abul|Abi|Ummu|Ibnu)"
_POST_BIN = r"(?:" + _COMPOUND + r"\s+" + _ATOM + r"|" + _ATOM + r")"
_NASAB = r"\s+(?:bin|binti)\s+" + _POST_BIN
#
# Nama lengkap: harus punya compound prefix (Abu/Ummu/...) ATAU minimal satu nasab (bin/binti)
_PERSON_BIN_RE = re.compile(
    r"\b((?:Abu|Ummu|Ibnu|Ibnul|Abul)\s+" + _ATOM + r"(?:" + _NASAB + r")*"
    r"|" + _ATOM + r"(?:" + _NASAB + r")+)"
)

# Pola nama terpotong: berakhir dengan "bin Al", "bin Abu", dsb.
_TRUNCATED_SUFFIX_RE = re.compile(
    r"\s+(?:bin|binti)\s+(?:Al|An|Ash|As|Ats|Ad|Ar|Az|At|Adz|Abu|Abul|Abi|Abdul)$"
)
# Pola untuk memperluas nama terpotong dari teks berikutnya
# \s* setelah hyphen menangani artefak OCR seperti "An- Nu'man"
_EXTEND_RE = re.compile(
    r"(-\s*[A-Z][a-z']+(?:'[a-z]*)?|\s+[A-Z][a-z']+(?:'[a-z]*)?)"
)

## 1.3 EVENT Patterns

In [3]:
EVENT_EXACT = [
    # Perang
    "Perang Badr", "Perang Badar",
    "Perang Uhud",
    "Perang Khandaq", "Perang Ahzab", "Perang Al-Khandaq",
    "Perang Khaibar",
    "Perang Hunain",
    "Perang Tabuk",
    "Perang Mu'tah",
    "Perang Hamra'ul Asad",
    # Penaklukan
    "Fathu Makkah", "Penaklukan Makkah",
    # Perjanjian
    "Perjanjian Hudaibiyah",
    "Piagam Madinah",
    # Baiat
    "Baiat Aqabah", "Baiat Aqabah Pertama", "Baiat Aqabah Kedua",
    "Baiat Aqabah Kubra",
    # Hijrah
    "Hijrah", "Hijrah ke Habasyah", "Hijrah ke Madinah",
    # Isra Mi'raj
    "Isra'", "Mi'raj", "Isra' dan Mi'raj", "Isra' Mi'raj",
    # Haji
    "Haji Wada'", "Haji Wada",
    # Lainnya
    "Futuh Makkah",
    "Pemboikotan di Syi'b Abu Thalib",
    "Tahun Duka Cita", "Tahun Kesedihan",
    "Nuzulul Quran",
    "Fathul Makkah",
]

# Pattern generik perang/ghazwah/sariyah
_EVENT_PERANG_RE = re.compile(
    r"\b(Perang\s+[A-Z][a-z']+(?:\s+[A-Z][a-z']+)?)"
)
_EVENT_GHAZWAH_RE = re.compile(
    r"\b(Ghazwah\s+[A-Z][a-z']+(?:\s+[A-Z][a-z']+)?)"
)

## 1.4 LOCATION Patterns

In [4]:
LOCATION_EXACT = [
    # Kota utama
    "Makkah", "Madinah", "Yastrib", "Yatsrib",
    "Thaif", "Tha'if",
    "Habasyah", "Najran",
    # Tempat suci & landmark
    "Ka'bah", "Baitullah", "Masjidil Haram", "Baitul Maqdis",
    "Gua Hira", "Gua Hira'", "Gua Tsur",
    "Jabal Nur", "Jabal Uhud",
    "Bukit Shafa", "Bukit Marwah", "Shafa", "Marwah",
    "Sumur Zamzam", "Zamzam",
    "Arafah", "Muzdalifah", "Mina",
    "Aqabah",
    # Wilayah
    "Hijaz", "Syam", "Yaman", "Irak", "Najd",
    "Tihamah", "Palestina", "Mesir",
    "Habasyah", "Persia",
    "Jazirah Arab",
    # Tempat spesifik
    "Badr", "Uhud", "Khandaq", "Khaibar", "Hunain",
    "Hudaibiyah", "Tabuk",
    "Dzul Hulaifah", "Dzu Thuwa'",
    "Al-Jurf", "Al-Kudr",
    "Qudaid",
    "Wadi Nakhlah",
    "Babilonia", "Babilon",
    # Pasar
    "Ukazh", "Majinnah", "Dzil-Majaz",
    # Sungai
    "Nil", "Eufrat",
    # Tempat lain
    "Sidratul Muntaha", "Al-Baitul-Ma'mur",
    "Laut Merah",
]

## 1.5 TIME Patterns

In [5]:
# Bulan Hijriah
BULAN_HIJRIAH = [
    "Muharram", "Shafar", "Rabi'ul Awwal", "Rabi'ul Akhir",
    "Jumadil Ula", "Jumadil Akhir", "Jumada",
    "Rajab", "Sya'ban", "Ramadhan",
    "Syawwal", "Dzul Qa'dah", "Dzul Hijjah",
    "Dzul Qi'dah",
]

# Regex patterns untuk waktu
_TIME_PATTERNS = [
    # tahun X Hijriah/Masehi/Nubuwah/SM
    re.compile(r"\b(tahun\s+(?:ke[\s-]?)?\d+(?:\s+(?:Hijriyah|Hijriah|Masehi|SM|H|M|dari\s+nubuwah|setelah\s+hijrah|sebelum\s+hijrah))?)"),
    # tanggal X bulan
    re.compile(r"\b(tanggal\s+\d+\s+(?:dari\s+)?(?:bulan\s+)?\w+)"),
    # bulan + tahun
    re.compile(r"\b(bulan\s+(?:" + "|".join(BULAN_HIJRIAH) + r")(?:\s+(?:tahun\s+)?\d+\s*(?:H|Hijriyah|Hijriah)?)?)"),
    # hari Senin/Selasa/dll
    re.compile(r"\b(hari\s+(?:Senin|Selasa|Rabu|Kamis|Jumat|Sabtu|Minggu|Tasyriq|kurban|tarwiyah))"),
    # X tahun sebelum/setelah
    re.compile(r"\b(\d+\s+tahun\s+(?:sebelum|setelah|sesudah)\s+\w+)"),
    # malam tanggal X
    re.compile(r"\b(malam\s+tanggal\s+\d+\s+(?:dari\s+)?(?:bulan\s+)?\w+)"),
    # Lailatul Qadr
    re.compile(r"\b(Lailatul[\s-]Qadr)"),
    re.compile(r"\b(Lailatul[\s-]Qadar)"),
    # pertengahan hari Tasyriq
    re.compile(r"\b(pertengahan\s+hari[\s-]hari\s+Tasyriq)"),
]

## 1.6 Matching Engine

In [6]:
def find_exact_matches(text, patterns, label):
    """Cari semua kemunculan pattern eksak di teks."""
    results = []
    for pat in patterns:
        # Escape regex special chars in pattern
        escaped = re.escape(pat)
        for m in re.finditer(r"\b" + escaped + r"\b", text):
            results.append({
                "entity_text": m.group(0),
                "label": label,
                "start_char": m.start(),
                "end_char": m.end(),
            })
    return results


def find_regex_matches(text, regex_list, label):
    """Cari semua kemunculan regex pattern di teks."""
    results = []
    for rx in regex_list:
        for m in rx.finditer(text):
            grp = 1 if m.lastindex else 0
            results.append({
                "entity_text": m.group(grp),
                "label": label,
                "start_char": m.start(grp),
                "end_char": m.end(grp),
            })
    return results


_INDO_STOPWORDS = {
    "Kemudian", "Lalu", "Wahai", "Maka", "Setelah", "Ketika", "Dengan",
    "Tentang", "Adapun", "Namun", "Sedangkan", "Menurut", "Bahkan",
    "Kepada", "Karena", "Terhadap", "Sementara", "Begitu", "Akhirnya",
    "Oleh", "Untuk", "Dalam", "Pada", "Dari", "Seperti", "Hingga",
    "Tanpa", "Selain", "Sebelum", "Sesudah", "Sambil", "Seraya",
    "Tatkala", "Tiba", "Saat", "Demi", "Bersama", "Bukan", "Tetapi",
    "Akan", "Jika", "Bila", "Walau", "Sekalipun", "Supaya", "Agar",
    "Antara", "Sekitar", "Berkata", "Mereka", "Merasa",
    "Sebab", "Padahal", "Bahwa", "Yakni", "Yaitu", "Rupanya",
    "Tanya", "Sesungguhnya", "Sungguh", "Sebenarnya", "Malah", "Justru", "Apalagi",
}


def find_person_bin(text):
    """Deteksi nama dengan 'bin/binti' yang belum tercakup di list eksak."""
    results = []
    for m in _PERSON_BIN_RE.finditer(text):
        name = m.group(1).strip()
        end_pos = m.end()

        if len(name) <= 5:  # skip yang terlalu pendek
            continue

        # Skip jika kata pertama adalah stopword bahasa Indonesia
        first_word = name.split()[0]
        if first_word in _INDO_STOPWORDS:
            continue

        # Perbaiki nama terpotong: "Ka'b bin Al" → "Ka'b bin Al-Khaththab"
        if _TRUNCATED_SUFFIX_RE.search(name):
            rest = text[end_pos:]
            ext = _EXTEND_RE.match(rest)
            if ext:
                name += ext.group(1)
                end_pos += ext.end()
                # Coba extend sekali lagi untuk compound seperti
                # "bin Abdul" + " " + "Muththalib"
                rest2 = text[end_pos:]
                ext2 = _EXTEND_RE.match(rest2)
                if ext2 and not ext2.group(1).startswith("-"):
                    name += ext2.group(1)
                    end_pos += ext2.end()

        results.append({
            "entity_text": name.strip(),
            "label": "PERSON",
            "start_char": m.start(),
            "end_char": end_pos,
        })
    return results


def find_time_bulan(text):
    """Deteksi nama bulan Hijriah di teks."""
    results = []
    for bulan in BULAN_HIJRIAH:
        escaped = re.escape(bulan)
        for m in re.finditer(r"\b" + escaped + r"\b", text):
            results.append({
                "entity_text": m.group(0),
                "label": "TIME",
                "start_char": m.start(),
                "end_char": m.end(),
            })
    return results


def deduplicate_entities(entities):
    """
    Hapus duplikat dan overlap.
    Jika ada overlap, pilih yang lebih panjang (lebih spesifik).
    """
    if not entities:
        return []

    # Urutkan: paling panjang dulu, lalu posisi awal
    entities.sort(key=lambda e: (-len(e["entity_text"]), e["start_char"]))

    kept = []
    used_spans = []

    for ent in entities:
        s, e = ent["start_char"], ent["end_char"]
        # Cek apakah overlap dengan span yang sudah diambil
        overlap = False
        for us, ue in used_spans:
            if s < ue and e > us:  # overlap
                overlap = True
                break
        if not overlap:
            kept.append(ent)
            used_spans.append((s, e))

    # Sort by position
    kept.sort(key=lambda x: x["start_char"])
    return kept


def _normalize_quotes(text):
    """Normalisasi karakter kutip unicode ke ASCII standar."""
    text = text.replace('\u2018', "'").replace('\u2019', "'")  # ' '
    text = text.replace('\u201C', '"').replace('\u201D', '"')  # " "
    text = text.replace('\u0060', "'")  # `
    text = text.replace('\u00B4', "'")  # ´
    return text


def extract_entities(text):
    """Ekstrak semua entitas dari teks."""
    if not isinstance(text, str) or not text.strip():
        return []
    text = _normalize_quotes(text)

    all_ents = []

    # 1. PERSON - exact match (paling panjang dulu)
    persons_sorted = sorted(PERSON_EXACT, key=len, reverse=True)
    all_ents.extend(find_exact_matches(text, persons_sorted, "PERSON"))

    # 2. PERSON - bin/binti pattern
    all_ents.extend(find_person_bin(text))

    # 3. EVENT - exact match
    events_sorted = sorted(EVENT_EXACT, key=len, reverse=True)
    all_ents.extend(find_exact_matches(text, events_sorted, "EVENT"))

    # 4. EVENT - Perang/Ghazwah regex
    all_ents.extend(find_regex_matches(text, [_EVENT_PERANG_RE, _EVENT_GHAZWAH_RE], "EVENT"))

    # 5. LOCATION - exact match
    locations_sorted = sorted(LOCATION_EXACT, key=len, reverse=True)
    all_ents.extend(find_exact_matches(text, locations_sorted, "LOCATION"))

    # 6. TIME - regex patterns
    all_ents.extend(find_regex_matches(text, _TIME_PATTERNS, "TIME"))

    # 7. TIME - bulan Hijriah
    all_ents.extend(find_time_bulan(text))

    # 8. Post-processing: bersihkan noise prefix yang lolos
    cleaned = []
    for ent in all_ents:
        name = ent["entity_text"]
        first_word = name.split()[0] if name else ""
        if first_word in _INDO_STOPWORDS and " " in name:
            # Strip prefix noise, perbaiki start_char
            stripped = name[len(first_word):].strip()
            ent["start_char"] += len(name) - len(stripped)
            ent["entity_text"] = stripped
        cleaned.append(ent)

    # Deduplicate
    return deduplicate_entities(cleaned)

---
## 1.7 Jalankan Pipeline

In [7]:
# 1. Baca seed CSV
df = pd.read_csv(IN_SEED, sep=";", encoding="utf-8-sig").fillna("")
df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()
print(f"Chunks dibaca: {len(df)}")

# Kolom dasar
base_cols = ["chunk_id", "doc_id", "chunk_index", "judul_bab", "judul_sub_bab", "halaman", "teks_chunk"]

# 2. Proses setiap chunk
rows = []
total_entities = 0
chunks_with_ents = 0

for _, row in df.iterrows():
    text = str(row.get("teks_chunk", ""))
    entities = extract_entities(text)

    base = {col: row.get(col, "") for col in base_cols}

    if entities:
        chunks_with_ents += 1
        for ent in entities:
            r = {**base}
            r["entity_text"] = ent["entity_text"]
            r["label"] = ent["label"]
            r["notes"] = ""
            r["start_char"] = ent["start_char"]
            r["end_char"] = ent["end_char"]
            rows.append(r)
            total_entities += 1
    else:
        # Chunk tanpa entitas: tetap masukkan 1 baris kosong
        r = {**base}
        r["entity_text"] = ""
        r["label"] = ""
        r["notes"] = ""
        r["start_char"] = ""
        r["end_char"] = ""
        rows.append(r)

# 3. Buat DataFrame output
out_cols = base_cols + ["entity_text", "label", "notes", "start_char", "end_char"]
df_out = pd.DataFrame(rows, columns=out_cols)

# 4. Statistik
print(f"\nTotal entitas terdeteksi: {total_entities}")
print(f"Chunks dengan entitas: {chunks_with_ents}/{len(df)}")

label_counts = df_out[df_out["label"] != ""]["label"].value_counts()
print(f"\nDistribusi label:")
for label, count in label_counts.items():
    print(f"  {label}: {count}")

# 5. Simpan
df_out.to_csv(OUT_PRE, index=False, sep=";", encoding="utf-8-sig")
print(f"\nOutput disimpan ke: {OUT_PRE}")

# 6. Preview
sample = df_out[df_out["label"] != ""].head(15)
print(f"\nPreview 15 entitas pertama:")
for _, r in sample.iterrows():
    print(f"  [{r['label']:8s}] {r['entity_text']:<35s} (chunk: {r['chunk_id']})")

Chunks dibaca: 844

Total entitas terdeteksi: 6000
Chunks dengan entitas: 798/844

Distribusi label:
  PERSON: 4068
  LOCATION: 1434
  TIME: 307
  EVENT: 191

Output disimpan ke: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\manual_labelling\sirah_prelabelled.csv

Preview 15 entitas pertama:
  [TIME    ] bulan Dzul Qi'dah                   (chunk: 000354-001)
  [TIME    ] Dzul Hijjah                         (chunk: 000354-001)
  [PERSON  ] Rasulullah                          (chunk: 000354-001)
  [PERSON  ] Abu Bakar Ash-Shiddiq               (chunk: 000354-001)
  [PERSON  ] Ali bin Abu Thalib                  (chunk: 000354-001)
  [PERSON  ] Ali                                 (chunk: 000354-001)
  [PERSON  ] Abu Bakar                           (chunk: 000354-001)
  [PERSON  ] Abu Bakar                           (chunk: 000354-001)
  [PERSON  ] Ali                                 (chunk: 000354-001)
  [PERSON  ] Abu Bakar                           (ch